In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
!uv pip install --python /content/poseenv/bin/python \
    mmpose==1.3.2 --no-deps

Using Python 3.10.22 environment at: poseenv
Resolved 1 package in 324ms
Prepared 1 package in 306ms
Installed 1 package in 66ms
 + mmpose==1.3.2


In [17]:
import torch
import cv2
import os
import numpy as np
import pandas as pd

from pathlib import Path
from PIL import Image, ImageDraw
from tqdm import tqdm
from mmpose.apis import init_model, inference_topdown

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: CUDA not available (is torch a +cpu build?)")

Using device: cuda:0


In [40]:
import sys, types

stub = types.ModuleType("mmcv.ops")
stub.__path__ = []

def _missing(name):
    if name.startswith("__"):
        raise AttributeError(name)
    class _Unavailable:
        def __init__(self, *a, **k):
            raise RuntimeError(f"mmcv.ops.{name} is unavailable (mmcv has no compiled ops)")
    _Unavailable.__name__ = name
    return _Unavailable

stub.__getattr__ = _missing
sys.modules["mmcv.ops"] = stub

In [18]:
def _isArrayLike(obj):

    return not type(obj) == str and hasattr(obj, '__iter__') and hasattr(obj, '__len__')

In [38]:
class Databuilder:
    """
    YOLO cow detection -> dynamic crop around the cow -> resize to a COMMON size -> MMPose keypoints.

    width, height : size every saved crop is resized to (also sets the crop aspect ratio, so a
                    square output gives a square crop and nothing is stretched).
    pad           : margin added on each side of the YOLO box, as a fraction of the box size.
    size_mode     : "frame" -> crop size follows the cow in every frame (cow always fills the crop).
                    "video" -> one crop size per video (largest cow in the video + pad); the crop
                               still follows the cow's position but its scale is constant, which keeps
                               keypoint units consistent across frames (better for temporal models).
    overwrite     : re-run a video even if its crop annotation CSV already exists.
    """

    def __init__(
        self, img_file, save_root, width, height,
        load_model=True, pad=0.15, size_mode="frame", overwrite=False,
        device=None, yolo_batch=16
    ):
        assert size_mode in ("frame", "video"), "size_mode must be 'frame' or 'video'"
        self.img = img_file
        self.save_root = Path(save_root)
        self.crop_w = int(width)    # output size
        self.crop_h = int(height)
        self.pad = float(pad)
        self.size_mode = size_mode
        self.overwrite = overwrite
        self.device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        self.yolo_batch = int(yolo_batch)

        self.set_save_root(save_root)

        self.model = None
        self.pose_model = None

        if load_model:
            self.load_yolo()

    def set_save_root(self, save_root):
        """Point all output folders at a new root (lets one loaded model be reused across videos)."""
        self.save_root = Path(save_root)
        self.crop_save = self.save_root / "annotation" / "crop"
        self.keypoints_save = self.save_root / "annotation" / "keypoints"
        self.img_save = self.save_root / "crop_img"
        self.keypoints_vis = self.save_root / "keypoint_img"

        for folder in (
            self.crop_save,
            self.keypoints_save,
            self.img_save,
            self.keypoints_vis
        ):
            folder.mkdir(parents=True, exist_ok=True)

    def load_yolo(self):
        print("Loading YOLOv5...")

        self.model = torch.hub.load(
            r"/content/drive/MyDrive/hackathon/Data/yolov5",
            "yolov5x",
            source="local",
            device=self.device
        )

        self.model.conf = 0.01
        self.model.iou = 0.45
        self.model.classes = [19]  # Cow
        self.model.multi_label = False

    def load_pose(self):
        if self.pose_model is not None:
            return

        root = Path(r"/content/drive/MyDrive/hackathon/Data/mmpose")

        config = root / (
            "configs/animal_2d_keypoint/topdown_heatmap/"
            "animalpose/"
            "td-hm_hrnet-w32_8xb64-210e_animalpose-256x256.py"
        )

        checkpoint = root / "checkpoints" / "animalpose.pth"

        for path in (config, checkpoint):
            if not path.is_file():
                raise FileNotFoundError(f"Missing: {path}")

        print("Loading MMPose AnimalPose...")

        self.pose_model = init_model(
            str(config),
            str(checkpoint),
            device=self.device
        )

    # ------------------------------------------------------------------
    # Crop geometry
    # ------------------------------------------------------------------
    def _crop_size(self, bw, bh):
        """(w, h) of a crop with the output aspect ratio that contains the padded YOLO box."""
        aspect = self.crop_w / self.crop_h
        need_w = bw * (1 + 2 * self.pad)
        need_h = bh * (1 + 2 * self.pad)
        cw = max(need_w, need_h * aspect)
        return cw, cw / aspect

    @staticmethod
    def _place(center, size, limit):
        """Top-left coordinate along one axis. Shifted to stay inside the image when the crop fits;
        if the crop is larger than the image it is centred and the overhang is padded black."""
        start = center - size / 2
        if size <= limit:
            start = np.clip(start, 0, limit - size)
        return int(round(start))

    def crop_images(self, img_dir, out_dir):
        if self.model is None:
            self.load_yolo()

        img_dir = Path(img_dir)
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        csv_path = self.crop_save / f"{img_dir.name}.csv"
        if csv_path.exists() and not self.overwrite:
            print(f"Crops already exist for {img_dir.name} (overwrite=False) - skipping")
            return

        img_files = sorted(
            p for p in img_dir.iterdir()
            if p.suffix.lower() in (".jpg", ".jpeg", ".png")
        )

        # ---- pass 1: detect the cow in every frame ----
        dets = []
        bs = max(1, self.yolo_batch)
        for s in tqdm(range(0, len(img_files), bs), desc="YOLO detection"):
            batch = img_files[s:s + bs]
            results = self.model([str(p) for p in batch])   # batched GPU inference
            det_list = results.pandas().xyxy                # one DataFrame per image

            for img_path, detections in zip(batch, det_list):
                if detections.empty:
                    print(f"No cow detected: {img_path.name}")
                    continue

                # Select highest-confidence cow
                bb = detections.loc[detections["confidence"].idxmax()]

                with Image.open(img_path) as im:
                    width, height = im.size

                bw = float(bb["xmax"] - bb["xmin"])
                bh = float(bb["ymax"] - bb["ymin"])
                cw, ch = self._crop_size(bw, bh)
                dets.append({
                    "path": img_path, "bb": bb, "W": width, "H": height,
                    "cx": float((bb["xmin"] + bb["xmax"]) / 2),
                    "cy": float((bb["ymin"] + bb["ymax"]) / 2),
                    "cw": cw, "ch": ch,
                })

        if not dets:
            return

        # one crop size for the whole video (position still follows the cow)
        if self.size_mode == "video":
            cw_v = max(d["cw"] for d in dets)
            for d in dets:
                d["cw"], d["ch"] = cw_v, cw_v * self.crop_h / self.crop_w

        # ---- pass 2: crop, resize to the common size, save ----
        annotations = []
        for d in tqdm(dets, desc="Cropping"):
            cw, ch = int(round(d["cw"])), int(round(d["ch"]))
            x0 = self._place(d["cx"], cw, d["W"])
            y0 = self._place(d["cy"], ch, d["H"])

            with Image.open(d["path"]) as im:
                im = im.convert("RGB")
                # PIL pads out-of-image areas with black, so the geometry stays exact
                cropped = im.crop((x0, y0, x0 + cw, y0 + ch))
                cropped = cropped.resize((self.crop_w, self.crop_h), Image.LANCZOS)
                cropped.save(out_dir / d["path"].name)

            bb = d["bb"]
            annotations.append({
                "image": d["path"].name,
                "xmin": float(bb["xmin"]),
                "ymin": float(bb["ymin"]),
                "xmax": float(bb["xmax"]),
                "ymax": float(bb["ymax"]),
                "confidence": float(bb["confidence"]),
                "xcrop": x0,          # crop top-left in the ORIGINAL frame
                "ycrop": y0,
                "crop_w": cw,         # crop size in the ORIGINAL frame (before resizing)
                "crop_h": ch,
            })

        df = pd.DataFrame(annotations)
        name = img_dir.name

        df.to_csv(csv_path, index=False)
        df.to_hdf(
            self.crop_save / f"{name}.h5",
            key="df_with_missing",
            mode="w"
        )

    # ------------------------------------------------------------------
    # Pose
    # ------------------------------------------------------------------
    def _crop_lookup(self, name):
        """image name -> (xcrop, ycrop, crop_w, crop_h) from the crop annotation, if available."""
        p = self.crop_save / f"{name}.csv"
        if not p.is_file():
            return {}
        c = pd.read_csv(p)
        if "crop_w" not in c.columns:   # CSV from the old fixed-size version
            return {}
        return {r.image: (r.xcrop, r.ycrop, r.crop_w, r.crop_h) for r in c.itertuples()}

    def run_pose(self, crop_dir, pose_vis=True):
        self.load_pose()

        crop_dir = Path(crop_dir)
        vis_dir = self.keypoints_vis / crop_dir.name
        vis_dir.mkdir(parents=True, exist_ok=True)

        crop_geo = self._crop_lookup(crop_dir.name)

        images = sorted(
            p for p in crop_dir.iterdir()
            if p.suffix.lower() in (".jpg", ".jpeg", ".png")
        )

        records = []

        for img_path in tqdm(images, desc="Pose estimation"):
            with Image.open(img_path) as im:
                width, height = im.size

            bbox = np.array(
                [[0, 0, width, height]],
                dtype=np.float32
            )

            results = inference_topdown(
                self.pose_model,
                str(img_path),
                bboxes=bbox,
                bbox_format="xyxy"
            )

            if not results:
                continue

            pred = results[0].pred_instances
            keypoints = pred.keypoints[0]
            scores = pred.keypoint_scores[0]

            # map crop coordinates back to the original frame
            geo = crop_geo.get(img_path.name)

            for i, ((x, y), score) in enumerate(
                zip(keypoints, scores)
            ):
                x, y = float(x), float(y)
                if geo is not None:
                    x0, y0, cw, ch = geo
                    x_full = x0 + x * cw / width
                    y_full = y0 + y * ch / height
                else:
                    x_full = y_full = float("nan")

                records.append({
                    "image": img_path.name,
                    "keypoint": i,
                    "x": x,               # crop coordinates (resized crop)
                    "y": y,
                    "score": float(score),
                    "x_full": x_full,     # original-frame coordinates
                    "y_full": y_full,
                })

            if pose_vis:
                image = Image.open(img_path).convert("RGB")
                draw = ImageDraw.Draw(image)

                # Draw skeleton connections if metadata provides them
                links = self.pose_model.dataset_meta.get(
                    "skeleton_links", []
                )

                for a, b in links:
                    if scores[a] < 0.3 or scores[b] < 0.3:
                        continue

                    draw.line(
                        [
                            tuple(map(float, keypoints[a])),
                            tuple(map(float, keypoints[b]))
                        ],
                        fill="lime",
                        width=3
                    )

                # Draw keypoints
                for i, ((x, y), score) in enumerate(
                    zip(keypoints, scores)
                ):
                    if score < 0.3:
                        continue

                    x, y = float(x), float(y)
                    r = 5

                    draw.ellipse(
                        (x-r, y-r, x+r, y+r),
                        fill="red"
                    )

                    draw.text(
                        (x+7, y+7),
                        str(i),
                        fill="yellow"
                    )

                output_path = vis_dir / (
                    img_path.stem + "_pose.png"
                )
                image.save(output_path)

        if records:
            df = pd.DataFrame(records)
            name = crop_dir.name

            df.to_csv(
                self.keypoints_save / f"{name}.csv",
                index=False
            )
            df.to_hdf(
                self.keypoints_save / f"{name}.h5",
                key="keypoints",
                mode="w"
            )

            print(f"Saved keypoints for {name}")

    def pipeline(
        self, needcrop=True, pose=True, pose_vis=True
    ):
        image_dirs = (
            [self.img]
            if isinstance(self.img, (str, Path))
            else self.img
        )

        for img_dir in image_dirs:
            img_dir = Path(img_dir)

            if not img_dir.is_dir():
                raise NotADirectoryError(img_dir)

            crop_dir = self.img_save / img_dir.name

            if needcrop:
                self.crop_images(img_dir, crop_dir)

            if pose:
                if not crop_dir.is_dir():
                    print(f"No cropped images: {crop_dir}")
                    continue

                self.run_pose(crop_dir, pose_vis)

In [20]:
import cv2
import os

def video_to_frames(video_path, output_folder="test_images", target_fps=15):
    os.makedirs(output_folder, exist_ok=True)

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    original_fps = cap.get(cv2.CAP_PROP_FPS)

    if original_fps <= 0:
        raise ValueError("Could not determine video FPS")

    print(f"Original FPS: {original_fps}")
    print(f"Target FPS: {target_fps}")

    frame_number = 0
    saved_number = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        current_time = frame_number / original_fps
        expected_saved_frames = int(current_time * target_fps) + 1

        if saved_number < expected_saved_frames:
            frame_name = f"{saved_number:06d}.jpg"
            frame_path = os.path.join(output_folder, frame_name)

            cv2.imwrite(frame_path, frame)
            saved_number += 1

        frame_number += 1

    cap.release()

    print(f"Done! Saved {saved_number} frames at {target_fps} FPS.")

In [31]:
pwd

'/content/drive/MyDrive/hackathon/Data'

In [32]:
home = Path(r"/content/drive/MyDrive/hackathon/Data")

lame = Path(str(Path) + r"\Lame")
normal = Path(str(Path) + r"\Normal")

lame = home / "Lame"
normal = home / "Normal"

print(normal.is_dir())  # True
print(lame.is_dir())    # True

if True:
    for folder in [normal, lame]:
        for video in folder.glob("*.mp4"):

            output_folder = folder / video.stem
            output_folder.mkdir(parents=True, exist_ok=True)

            video_to_frames(
                video_path=str(video),
                output_folder=str(output_folder),
                target_fps=10
            )

True
True
Original FPS: 60.0
Target FPS: 10
Done! Saved 25 frames at 10 FPS.
Original FPS: 25.0
Target FPS: 10
Done! Saved 51 frames at 10 FPS.
Original FPS: 29.166666666666668
Target FPS: 10
Done! Saved 51 frames at 10 FPS.
Original FPS: 29.583333333333332
Target FPS: 10
Done! Saved 45 frames at 10 FPS.
Original FPS: 30.0
Target FPS: 10
Done! Saved 38 frames at 10 FPS.
Original FPS: 29.97
Target FPS: 10
Done! Saved 69 frames at 10 FPS.
Original FPS: 30.0
Target FPS: 10
Done! Saved 51 frames at 10 FPS.
Original FPS: 60.0
Target FPS: 10
Done! Saved 19 frames at 10 FPS.
Original FPS: 30.0
Target FPS: 10
Done! Saved 47 frames at 10 FPS.
Original FPS: 29.583333333333332
Target FPS: 10
Done! Saved 41 frames at 10 FPS.
Original FPS: 29.97002997002997
Target FPS: 10
Done! Saved 83 frames at 10 FPS.
Original FPS: 30.0
Target FPS: 10
Done! Saved 50 frames at 10 FPS.
Original FPS: 30.0
Target FPS: 10
Done! Saved 82 frames at 10 FPS.
Original FPS: 60.0
Target FPS: 10
Done! Saved 14 frames at 10 F

In [33]:
target_width = 360

if True:
    for category in [home / "Normal", home / "Lame"]:
        for folder in category.iterdir():

            if not folder.is_dir():
                continue

            for image_path in folder.glob("*.jpg"):
                img = cv2.imread(str(image_path))

                if img is None:
                    print(f"Could not read: {image_path}")
                    continue

                h, w = img.shape[:2]

                # Preserve aspect ratio
                new_height = round(h * target_width / w)

                # Use appropriate interpolation
                interpolation = (
                    cv2.INTER_AREA if w > target_width
                    else cv2.INTER_CUBIC
                )

                resized = cv2.resize(
                    img,
                    (target_width, new_height),
                    interpolation=interpolation
                )

                # Overwrite original frame
                cv2.imwrite(str(image_path), resized)

            print(f"Done: {folder}")

Done: /content/drive/MyDrive/hackathon/Data/Normal/N (1)
Done: /content/drive/MyDrive/hackathon/Data/Normal/N (11)
Done: /content/drive/MyDrive/hackathon/Data/Normal/N (12)
Done: /content/drive/MyDrive/hackathon/Data/Normal/N (13)
Done: /content/drive/MyDrive/hackathon/Data/Normal/N (14)
Done: /content/drive/MyDrive/hackathon/Data/Normal/N (16)
Done: /content/drive/MyDrive/hackathon/Data/Normal/N (17)
Done: /content/drive/MyDrive/hackathon/Data/Normal/N (18)
Done: /content/drive/MyDrive/hackathon/Data/Normal/N (19)
Done: /content/drive/MyDrive/hackathon/Data/Normal/N (2)
Done: /content/drive/MyDrive/hackathon/Data/Normal/N (20)
Done: /content/drive/MyDrive/hackathon/Data/Normal/N (21)
Done: /content/drive/MyDrive/hackathon/Data/Normal/N (22)
Done: /content/drive/MyDrive/hackathon/Data/Normal/N (23)
Done: /content/drive/MyDrive/hackathon/Data/Normal/N (24)
Done: /content/drive/MyDrive/hackathon/Data/Normal/N (25)
Done: /content/drive/MyDrive/hackathon/Data/Normal/N (3)
Done: /content/dr

In [41]:
import time
import shutil
import traceback
from pathlib import Path

# "data" lives next to this notebook
project_dir = Path.cwd()
data_out = project_dir / "data"
work_root = data_out / "_work"

for label in ["Lame", "Normal"]:
    (data_out / label).mkdir(parents=True, exist_ok=True)
work_root.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE, "| output:", data_out)

# load YOLO once and reuse it for every video
builder = Databuilder([], work_root, 256, 256, size_mode="video", device=DEVICE, yolo_batch=16)
builder.load_pose()   # crash immediately if the pose model can't load

failed = []

for label, folder in [("Normal", normal), ("Lame", lame)]:
    video_dirs = sorted(f for f in folder.iterdir() if f.is_dir())

    for i, video_dir in enumerate(video_dirs, 1):
        name = video_dir.name
        target_csv = data_out / label / f"{name}.csv"

        # resume support
        if target_csv.exists():
            print(f"[{label} {i}/{len(video_dirs)}] skip (done): {name}")
            continue

        print(f"[{label} {i}/{len(video_dirs)}] processing: {name}")
        start = time.time()

        try:
            builder.img = [str(video_dir)]
            builder.set_save_root(work_root / label / name)
            builder.pipeline(needcrop=True, pose=True, pose_vis=True)
        except Exception:
            print(f"FAILED: {label}/{name}")
            traceback.print_exc()
            failed.append((label, name))
            continue

        csv_path = builder.keypoints_save / f"{name}.csv"
        if not csv_path.is_file():
            print(f"No keypoints CSV found for {label}/{name}")
            failed.append((label, name))
            continue

        shutil.copy2(csv_path, target_csv)
        print(f"  saved -> {target_csv}  ({time.time() - start:.0f}s)")

print(f"\nFinished. Failed: {len(failed)}")
for label, name in failed:
    print(f"  {label}/{name}")

Device: cuda:0 | output: /content/drive/MyDrive/hackathon/Data/data
Loading YOLOv5...
YOLOv5 🚀 v7.0-570-gcb2f17fd Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
YOLOv5x summary: 256 layers, 86,705,005 parameters, 0 gradients, 205.4 GFLOPs
Adding AutoShape... 
Loading MMPose AnimalPose...
Loads checkpoint by local backend from path: /content/drive/MyDrive/hackathon/Data/mmpose/checkpoints/animalpose.pth
[Normal 1/38] processing: N (1)
Crops already exist for N (1) (overwrite=False) - skipping


Pose estimation: 100%|██████████| 25/25 [00:03<00:00,  7.34it/s]


Saved keypoints for N (1)
  saved -> /content/drive/MyDrive/hackathon/Data/data/Normal/N (1).csv  (4s)
[Normal 2/38] processing: N (11)
Crops already exist for N (11) (overwrite=False) - skipping


Pose estimation: 100%|██████████| 51/51 [00:08<00:00,  6.37it/s]


Saved keypoints for N (11)
  saved -> /content/drive/MyDrive/hackathon/Data/data/Normal/N (11).csv  (8s)
[Normal 3/38] processing: N (12)
Crops already exist for N (12) (overwrite=False) - skipping


Pose estimation: 100%|██████████| 51/51 [00:06<00:00,  7.32it/s]


... (237 outputs left out)

Pose estimation: 100%|██████████| 57/57 [00:08<00:00,  6.77it/s]


Saved keypoints for L (7)
  saved -> /content/drive/MyDrive/hackathon/Data/data/Lame/L (7).csv  (14s)
[Lame 44/44] processing: L (8)


Pose estimation: 100%|██████████| 60/60 [00:09<00:00,  6.16it/s]


Saved keypoints for L (8)
  saved -> /content/drive/MyDrive/hackathon/Data/data/Lame/L (8).csv  (15s)

Finished. Failed: 1
  Normal/N (32)


In [37]:
%pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 58.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics]
